# oscNext L4 — pybdt ile uçtan uca

L3 `.i3` dosyalarından eğitilmiş bir BDT'ye kadar tüm süreç.

**Motor: pybdt** (IceCube'un kendi AdaBoost kütüphanesi). Bu, teknik notun
(v00.074, §3.6.1 — LightGBM) resmi yönteminden **bilinçli** bir sapmadır;
gerekçesi `CLAUDE.md`'de.

## Çalıştırma ortamı

Jupyter, pybdt'nin derlendiği build'in env-shell'i içinden başlatılmış olmalı:

```bash
eval $(/cvmfs/icecube.opensciencegrid.org/py3-v4.4.2/setup.sh)
cd /data/user/$(whoami)/icetray_build/build && ./env-shell.sh
cd ~/l4 && python -m jupyter lab --no-browser --port=8896
```

## Ağır iş `.py` dosyalarında

Bu notebook ince bir arayüz. Mantık iki modülde:

| Modül | İçerik |
|---|---|
| `l4_run.py` | `process_L4.py` sürücüsü + canlı ilerleme çubuğu |
| `l4_data.py` | `REGISTRY`, HDF5→numpy yükleme, ağırlıklar, tutarlılık kontrolleri |

Modüller `git pull` ile güncellenir. Notebook'ta bir şey bozulursa
**Kernel → Restart** yeter — hücreyi düzenlemek hafızadaki eski tanımı
değiştirmez.

| # | Ne yapar | Sıklık |
|---|---|---|
| 0 | Konfigürasyon + ortam kontrolü | her açılışta |
| 1 | L3 → L4 işleme | bir kez, uzun sürer |
| 2 | Booking doğrulaması | bir kez |
| 3 | Feature registry + tutarlılık | değişken seçimi değişince |
| 4 | HDF5 → numpy | sık |
| 5 | Ağırlıklar | sık |
| 6 | pybdt DataSet + train/test | sık |
| 7 | Eğitim | sık |
| 8 | Doğrulama (overtraining) | sık |
| 9 | Kesim seçimi | sık |
| 10 | Modeli frame'e uygulama | en son |

## 0. Konfigürasyon ve ortam kontrolü

`pybdt` ve `tables` zorunlu. `pandas` **kullanılmıyor** — IceTray ortamında
bulunmayabilir.

In [ ]:
import os, sys, glob, json, shlex, subprocess, time
import numpy as np

# --- zorunlu: pybdt ---
try:
    from pybdt import ml, util
    print("pybdt        OK  ", os.path.dirname(ml.__file__))
except ImportError as e:
    raise SystemExit(
        "pybdt import edilemedi (%s).\n"
        "Jupyter'i pybdt build'inin env-shell'inden baslatin -- bkz. README.\n"
        "NOT: dogru import 'import pybdt', 'from icecube import pybdt' DEGIL." % e)

# --- zorunlu: pytables ---
try:
    import tables
    print("tables       OK  ", tables.__version__)
except ImportError:
    raise SystemExit("pytables yok -- HDF5 okunamaz.")

# --- opsiyonel ---
_NOTE = {"matplotlib": "grafik cizilemez, pybdt.validate calismaz",
         "scipy":      "pybdt.validate calismaz",
         "simweights": "CORSIKA agirligi yaklasik olur",
         "ipywidgets": "ilerleme cubugu ASCII'ye duser"}
for _n in _NOTE:
    try:
        _m = __import__(_n)
        print("%-12s OK   %s" % (_n, getattr(_m, "__version__", "")))
    except ImportError:
        print("%-12s YOK  (%s)" % (_n, _NOTE[_n]))

import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "font.size": 9})

In [ ]:
# ---------------------------------------------------------------------------
# YOLLAR
# ---------------------------------------------------------------------------
# Scriptlerin (process_L4.py, pybdt_train.py, l4_*.py) bulundugu dizin
L4_CODE_DIR = os.environ.get("OSCNEXT_L4_CODE", ".")
if os.path.abspath(L4_CODE_DIR) not in sys.path:
    sys.path.insert(0, os.path.abspath(L4_CODE_DIR))

# Ciktilar.  HDF5 ONLARCA GB olabilir -- home dizininde kota varsa
# OSCNEXT_OUT_ROOT'u /data/user/$USER/... altina alin.
OUTPUT_ROOT = os.environ.get(
    "OSCNEXT_OUT_ROOT", os.path.join(os.path.abspath(L4_CODE_DIR), "L4_output"))

HDF_BASE  = os.path.join(OUTPUT_ROOT, "hdf5")     # process_L4.py ciktisi
DS_BASE   = os.path.join(OUTPUT_ROOT, "ds")       # pybdt DataSet dosyalari
MODEL_DIR = os.path.join(OUTPUT_ROOT, "models")   # .bdt + .validator + grafikler

for _d in (HDF_BASE, DS_BASE, MODEL_DIR):
    os.makedirs(_d, exist_ok=True)

PROCESS_PY = os.path.join(L4_CODE_DIR, "process_L4.py")
TRAIN_PY   = os.path.join(L4_CODE_DIR, "pybdt_train.py")

RNG_SEED = 12345
rng = np.random.default_rng(RNG_SEED)

_st = os.statvfs(OUTPUT_ROOT)
print("Cikti koku : %s" % OUTPUT_ROOT)
print("Bos disk   : %.1f GB" % (_st.f_bavail * _st.f_frsize / 1e9))
for _p in (PROCESS_PY, TRAIN_PY):
    print("%-16s %s" % (os.path.basename(_p), "var" if os.path.exists(_p) else "YOK!"))

## 1. L3 → L4 işleme

`process_L4.py`'yi her örnek için çalıştırır. **Uzun sürer**, bir kez yapılır —
HDF5'ler üretildikten sonra 2. bölümden devam edebilirsin.

`--apply-cut` **kullanılmıyor**: modeller eğitilmeden önce tüm olaylar book
edilmeli, yoksa eğitim setini kesmiş oluruz.

In [ ]:
GCD = "/cvmfs/icecube.opensciencegrid.org/data/GCD/GeoCalibDetectorStatus_IC86.All_Pass3.i3.gz"

# pass3 uretimi.  nutau ve gercek dedektor verisi bu uretimde YOK.
SAMPLES = {
    "nue":     dict(l3="/data/ana/LE/oscNext/pass3/genie/level3/23800/*.i3.zst",
                    flags=["--mc", "--genie"], kind="signal"),
    "numu":    dict(l3="/data/ana/LE/oscNext/pass3/genie/level3/23799/*.i3.zst",
                    flags=["--mc", "--genie"], kind="signal"),
    "corsika": dict(l3="/data/ana/LE/oscNext/pass3/corsika/level3/23694/*.i3.zst",
                    flags=["--corsika"], kind="muon_bg"),
    "noise":   dict(l3="/data/ana/LE/oscNext/pass3/noise/level3/23813/*.i3.zst",
                    flags=["--noise"], kind="noise_bg"),
}

for name, cfg in SAMPLES.items():
    cfg["hdf5"] = os.path.join(HDF_BASE, name, "L4_%s.hdf5" % name)
    cfg["n_l3_files"] = len(glob.glob(cfg["l3"]))
    print("%-8s %-10s %6d L3 dosyasi" % (name, cfg["kind"], cfg["n_l3_files"]))

In [ ]:
from l4_run import configure_runner, run_process, run_all
configure_runner(SAMPLES, PROCESS_PY, GCD)

### Önce smoke test

Tam üretime geçmeden tek dosyada 200 frame işleyip zincirin çalıştığını
doğrula.

> **`--n` FRAME sayar, olay değil.** Akışta G/C/D, Q ve P frame'leri var;
> P frame'lerin de ancak bir kısmı `InIceSplit`'e uyup L3 kesimini geçiyor.
> 200 frame → ~60 olay normal. Çıktı kademeyi gösteriyor.

In [ ]:
smoke = run_process("nue", n_frames=200)

### Tam üretim

`chunk_files=10` → her 10 L3 dosyası ayrı bir parça (`L4_nue_part000.hdf5`, …).

- **Gerçek yüzde ve ETA** — parça sayısı baştan belli.
- **Kaldığı yerden devam** — tamamlanan parçalar atlanır, çökme halinde
  sadece o parça kaybolur.

Bozuk `.i3.zst` dosyaları `--scan quick` (varsayılan) ile eleniyor; tray yine
patlarsa `--retries` o dosyayı atıp devam ediyor.

In [ ]:
results = run_all(chunk_files=10)

## 2. Booking doğrulaması

**İşleme bittikten sonra ilk iş bu.** Bir tray modülü sessizce başarısız
olursa tablo hiç yazılmaz; bunu üç bölüm sonra "bu değişken neden hep NaN"
diye keşfetmek yerine burada yakala.

In [ ]:
from l4_data import dump_tables

TABLES = dump_tables(SAMPLES["nue"]["hdf5"].replace(".hdf5", "_smoke.hdf5"))

In [ ]:
# Belirli bir tabloya bakmak icin, orn. iLineFit kolon adi:
# dump_tables(SAMPLES["nue"]["hdf5"].replace(".hdf5", "_smoke.hdf5"),
#             only=["iLineFit"])

## 3. Feature registry ve tutarlılık

`REGISTRY` her BDT değişkenini `(HDF5 tablosu, kolon)` çiftine bağlar.
`ALTS` isim varyasyonlarını çözer — meta-proje/pass sürümüne göre kolon
adları değişiyor, dosyada **gerçekten hangisi varsa** o kullanılır.

`check_feature_map()` `REGISTRY` ile `l4_classifier_module.FEATURE_MAP`
çakışıyor mu diye bakar. Eğitimde bir kolon, frame'e uygularken başka bir
kolon okunursa model **hata fırlatmadan** saçmalar — bu yüzden kodla
kontrol ediliyor (AST ile okuyor, icetray gerekmiyor).

In [ ]:
from l4_data import (REGISTRY, ALTS, AUX, NOISE_FEATURES, MUON_FEATURES,
                     WANTED, check_registry, check_feature_map)

print("--- noise BDT girdileri (Tablo 11) ---")
check_registry(TABLES, NOISE_FEATURES)
print("\n--- muon BDT girdileri (Tablo 12) ---")
check_registry(TABLES, MUON_FEATURES)
print("\n--- agirlik kolonlari (AUX) ---")
check_registry(TABLES, list(AUX))
print()
check_feature_map()

## 4. HDF5 → numpy

Tablolar `Run/Event/SubEvent` üzerinden eşleştiriliyor — satır sırasına
güvenilmiyor. Bir frame objesi bazı olaylarda yoksa sıraya dayalı okuma
**kayar** ve olay A'nın `cog_z`'si olay B'nin `NchCleaned`'i ile eşleşir.

Ağırlık böleni (`_n_files`) her HDF5'in yanındaki `.meta.json`'dan okunan
**L3 dosya sayısı**dır — HDF5 dosya sayısı değil.

In [ ]:
from l4_data import load_sample

data = {}
for name in SAMPLES:
    d = load_sample(name, SAMPLES, WANTED)
    if d is not None:
        data[name] = d

print("\nYuklendi:", {k: len(v["Run"]) for k, v in data.items()})

### Sağlık kontrolü

Bir değişken bir örnekte **tamamen** NaN'sa o kolon book edilmemiş demektir —
eğitime sokarsan model onu sessizce görmezden gelir.

In [ ]:
print("%-22s %s" % ("degisken", "  ".join("%9s" % s for s in data)))
for f in NOISE_FEATURES + MUON_FEATURES:
    row, bad = [], False
    for d in data.values():
        v = d.get(f)
        frac = 100.0 * np.mean(~np.isfinite(v)) if v is not None else 100.0
        bad |= frac > 99.9
        row.append("%8.1f%%" % frac)
    print("%-22s %s%s" % (f, "  ".join("%9s" % x for x in row),
                          "  <-- HEP EKSIK" if bad else ""))
print("\n(NaN yuzdesi.  %100 olan bir kolon book edilmemis demektir.)")

## 5. Ağırlıklar

Üç ayrı kavram:

1. **`w_phys` [Hz]** — fiziksel oran. Dağılım ve kesim performansı için.
2. **`w_train`** — eğitim ağırlığı (bölüm 6'da). Teknik notun ön işlemesi:
   sınıf toplamları eşitlenir, sonra 0–1 aralığına çekilir (§3.6.1).
3. **Ağırlıksız sayım** — istatistiksel yeterlilik.

`add_weights` sonucu teknik notun **Tablo 13** (L3 oranları) ile
karşılaştırıp mertebe sapmasını işaretler.

> `NORM`/`GAMMA` gerçek atmosferik akı **değil**, basit bir güç yasası. Mutlak
> oranlar birebir tutmaz; şekil karşılaştırması ve eğitim için yeterli.
> Gerçek akı için `nuflux` (Honda) + salınım gerekir.

In [ ]:
from l4_data import add_weights

add_weights(data)

In [ ]:
n = len(data)
fig, axes = plt.subplots(1, n, figsize=(3.2 * n, 2.8))
axes = np.atleast_1d(axes)
for ax, (name, d) in zip(axes, data.items()):
    w = d["w_phys"]; w = w[np.isfinite(w) & (w > 0)]
    if w.size == 0:
        ax.set_title("%s: agirlik yok" % name, fontsize=8); continue
    ax.hist(np.log10(w), bins=40, color="tab:blue")
    ax.set_title("%s\nmaks/toplam = %.1f%%" % (name, 100 * w.max() / w.sum()),
                 fontsize=8)
    ax.set_xlabel("log10(w_phys)", fontsize=7); ax.tick_params(labelsize=6)
plt.tight_layout(); plt.show()
print("maks/toplam > %5 ise tek bir olay orani domine ediyor.")

## 6. pybdt DataSet'leri ve train/test ayrımı

`pybdt.ml.DataSet` bir *dict of numpy arrays*. Her BDT için dört dosya:
`sig_train`, `sig_test`, `bg_train`, `bg_test`.

**Eğitim ağırlığı burada hesaplanıyor**: sınıf toplamları eşitlenir, sonra
tüm ağırlıklar `[0, 1]`'e çekilir (teknik not §3.6.1).

Ayrım olay bazında rastgele (%50/%50). Dosya bazında ayırmak daha
muhafazakâr olurdu ama örnek sayısı azken bu yeterli.

In [ ]:
TRAIN_FRAC = 0.5


def stack(samples, features):
    """Birden fazla ornegi tek bir {ad: dizi} sozluguna yig."""
    out = {f: np.concatenate([data[s][f] for s in samples]) for f in features}
    out["w_phys"] = np.concatenate([data[s]["w_phys"] for s in samples])
    return out


def make_datasets(tag, sig_samples, bg_samples, features):
    """Bir BDT icin dort .ds dosyasi uret."""
    sig = stack(sig_samples, features)
    bg  = stack(bg_samples,  features)

    # --- egitim agirligi: sinif toplamlarini esitle, sonra [0,1]'e cek ---
    ws, wb = sig["w_phys"].copy(), bg["w_phys"].copy()
    for w in (ws, wb):
        w[~np.isfinite(w) | (w < 0)] = 0.0
    if ws.sum() > 0: ws /= ws.sum()
    if wb.sum() > 0: wb /= wb.sum()
    scale = max(ws.max(), wb.max())
    if scale > 0:
        ws /= scale; wb /= scale

    paths = {}
    for part, d, w in (("sig", sig, ws), ("bg", bg, wb)):
        istrain = rng.random(len(w)) < TRAIN_FRAC
        for split, mask in (("train", istrain), ("test", ~istrain)):
            cols = {f: np.asarray(d[f], dtype=np.float64)[mask] for f in features}
            cols["weight"] = w[mask]                 # egitim agirligi
            cols["w_phys"] = np.asarray(d["w_phys"], dtype=np.float64)[mask]
            p = os.path.join(DS_BASE, "%s_%s_%s.ds" % (tag, part, split))
            util.save(ml.DataSet(cols), p)
            paths["%s_%s" % (part, split)] = p
            print("  %-18s %8d olay -> %s" % ("%s %s" % (part, split),
                                              mask.sum(), os.path.basename(p)))
    return paths


print("=== noise BDT  (sinyal = nue+numu, arkaplan = noise) ===")
DS_NOISE = make_datasets("L4_noise", ["nue", "numu"], ["noise"], NOISE_FEATURES)

print("\n=== muon BDT  (sinyal = nue+numu, arkaplan = corsika) ===")
DS_MUON = make_datasets("L4_muon", ["nue", "numu"], ["corsika"], MUON_FEATURES)

## 7. Eğitim

`pybdt_train.py`'yi çağırır — eğitim mantığı tek yerde kalsın diye burada
tekrar yazılmıyor.

Hiperparametreler pybdt'nin kendi örneğinden (IC79 νμ analizi).
**oscNext için optimize edilmemiş** — teknik notun Tablo 10'u LightGBM'e ait
ve pybdt'ye taşınamaz (`num_leaves`, `max_bin`, `lambda_*` karşılıkları yok).
8. bölümdeki overtraining kontrolüne bakarak ayarla.

In [ ]:
HYPER = ["--num-trees", "300", "--depth", "3", "--beta", "0.7",
         "--prune-strength", "35", "--frac-random-events", "0.5",
         "--use-purity"]


def run_train(name, ds, features, extra=()):
    # --features ACIKCA veriliyor: .ds icinde w_phys gibi BDT girdisi
    # OLMAYAN kolonlar da var, otomatik secime birakilmamali -- yoksa model
    # fiziksel agirligi bir degisken sanip ogrenir.
    cmd = [sys.executable, "-u", TRAIN_PY, "--name", name, "--outdir", MODEL_DIR,
           "--sig-train", ds["sig_train"], "--bg-train", ds["bg_train"],
           "--sig-test",  ds["sig_test"],  "--bg-test",  ds["bg_test"],
           "--features", ",".join(features)] + HYPER + list(extra)
    print("$ " + " ".join(shlex.quote(c) for c in cmd) + "\n")
    p = subprocess.run(cmd, capture_output=True, text=True)
    print(p.stdout)
    if p.returncode != 0:
        print("--- HATA ---\n" + (p.stderr or "")[-3000:])
    return p.returncode == 0


run_train("L4_noise", DS_NOISE, NOISE_FEATURES)

In [ ]:
run_train("L4_muon", DS_MUON, MUON_FEATURES)

## 8. Doğrulama

`pybdt_train.py` `.validator` dosyasını kaydetti — skorlar zaten hesaplanmış.

**Overtraining ölçütü**: pybdt'nin KS testi. `p_KS ≲ 0.01` ise overtraining
var (`pybdt/resources/docs/man_overtraining.rst`) → `--depth` düşür,
`--prune-strength` artır ya da `--min-split` büyüt.

In [ ]:
V, META = {}, {}
for name in ("L4_noise", "L4_muon"):
    try:
        V[name] = util.load(os.path.join(MODEL_DIR, "%s.validator" % name))
        META[name] = json.load(open(os.path.join(MODEL_DIR, "%s.json" % name)))
        m = META[name]["metrics"]
        flag = "" if min(m["ks_signal"], m["ks_background"]) > 0.01 else "  <-- OVERTRAINING"
        print("%-10s %d agac,  p_KS sinyal=%.4f  arkaplan=%.4f%s"
              % (name, META[name]["n_trees"], m["ks_signal"],
                 m["ks_background"], flag))
    except (FileNotFoundError, OSError):
        print("%-10s henuz egitilmedi" % name)

In [ ]:
from IPython.display import Image, display

for name in V:
    for kind in ("overtrain", "dist", "rate"):
        p = os.path.join(MODEL_DIR, "%s_%s.png" % (name, kind))
        if os.path.exists(p):
            print(name, kind)
            display(Image(filename=p))

## 9. Kesim seçimi

pybdt skoru LightGBM'in `P(sinyal)` olasılığıyla **aynı ölçekte değil** —
teknik nottaki 0.70 / 0.65 değerleri buraya taşınamaz. Kesimi kendi
rate-vs-cut eğrinden seç.

Referans hedefler (v00.07, pass2, Tablo 13):
- **noise**: 36.6 mHz → <0.3 mHz gürültü, nötrinoların ~%96'sı korunur
- **muon**: muonların %94'ü atılır, nötrinoların %87'si korunur

In [ ]:
def scan_cut(name, n=200):
    """Kesim degerine karsi sinyal verimi / arkaplan reddi (test seti)."""
    v, meta = V[name], META[name]
    expr = meta["score_expr"]

    s = v.eval("test_sig", expr)
    b = v.eval("test_bg",  expr)
    # Agirlik olarak FIZIKSEL agirlik: egitim agirligi siniflari esitlemek
    # icin olceklenmisti, fiziksel oran degil.
    try:
        ws = np.nan_to_num(v.eval("test_sig", "w_phys"))
        wb = np.nan_to_num(v.eval("test_bg",  "w_phys"))
    except Exception:
        print("  [!] w_phys yok -> egitim agirligi kullaniliyor, "
              "oranlar FIZIKSEL DEGIL")
        ws = v.get_values_weights("test_sig", expr)[1]
        wb = v.get_values_weights("test_bg",  expr)[1]

    cuts = np.linspace(min(s.min(), b.min()), max(s.max(), b.max()), n)
    eff = np.array([ws[s >= c].sum() for c in cuts]) / ws.sum()
    rej = 1 - np.array([wb[b >= c].sum() for c in cuts]) / wb.sum()

    fig, ax = plt.subplots(figsize=(5, 3.2))
    ax.plot(cuts, 100 * eff, label="sinyal verimi", color="tab:blue")
    ax.plot(cuts, 100 * rej, label="arkaplan reddi", color="tab:red")
    ax.set_xlabel("pybdt skoru"); ax.set_ylabel("%")
    ax.grid(alpha=.3); ax.legend(fontsize=8); ax.set_title(name, fontsize=10)
    plt.tight_layout(); plt.show()
    return cuts, eff, rej


TARGET_REJ = {"L4_noise": 0.99, "L4_muon": 0.94}
CUTS = {}
for name in V:
    cuts, eff, rej = scan_cut(name)
    i = np.argmin(np.abs(rej - TARGET_REJ.get(name, 0.94)))
    CUTS[name] = float(cuts[i])
    print("%s: red %%%.0f -> kesim %.3f,  sinyal verimi %%%.1f"
          % (name, 100 * TARGET_REJ.get(name, 0.94), cuts[i], 100 * eff[i]))

## 10. Modeli frame'e uygulama

`.i3` dosyalarını yeniden işleyip her frame'e BDT skorunu yazmak için:

```python
from pybdt_classifier_module import PyBDTClassifier

tray.Add(PyBDTClassifier, "noise_clf",
         ModelFile="<MODEL_DIR>/L4_noise.bdt",
         OutputKey="L4_NoiseClassifier_pybdt")

tray.Add(PyBDTClassifier, "muon_clf",
         ModelFile="<MODEL_DIR>/L4_muon.bdt",
         OutputKey="L4_MuonClassifier_pybdt")
```

> Modül değişkenleri frame'den `l4_classifier_module.FEATURE_MAP` üzerinden
> okur; `REGISTRY` ise HDF5 kolonunu tarif eder. İkisi ayrışırsa eğitimde bir
> şey, uygulamada başka bir şey okunur ve model **hata vermeden** yanlış
> sonuç üretir. Aşağıdaki kontrol bunu yakalar (bölüm 3'tekiyle aynı).

In [ ]:
check_feature_map()

## Kontrol listesi

**İşleme**
- [ ] Smoke test temiz, kademe makul (bölüm 1)
- [ ] `check_registry` tüm BDT girdilerini buldu (bölüm 3)
- [ ] `check_feature_map()` çakışma bulmadı (bölüm 3)
- [ ] Hiçbir değişken %100 NaN değil (bölüm 4)

**Ağırlıklar**
- [ ] `.meta.json` uyarısı çıkmadı (yoksa bölen yanlış → oranlar kayar)
- [ ] `maks/toplam < %5`
- [ ] Tablo 13 ile mertebe tutuyor (bölüm 5)

**Eğitim**
- [ ] `p_KS > 0.01` — overtraining yok (bölüm 8)
- [ ] Kesim seçildi, verim/red referans mertebelerde (bölüm 9)

---

**Hâlâ doğrulanmamış olanlar** — ayrıntı `CLAUDE.md` → *Açık riskler* ve
`TEKNIK_NOT_KARSILASTIRMA.md`:

- VICH'in COG'u artık teknik nota uygun (fiducial hitler) ama yük ağırlıklı
  olup olmadığı ve pulse serisi seçimi doğrulanmadı
- `accumulated_time`'ın referans zamanı (ilk pulse mü, tetikleme mi)
- `FullTimeLengthRatio`'nun yönü
- Muon BDT arka planı gerçek veri yerine CORSIKA → data/MC kontrolü yok
- ντ seti yok → sinyalin ~%3'ü eksik